<a href="https://colab.research.google.com/github/AICHUCKY/Ai-with-Chucky-Colab-Notebooks/blob/main/Yue2_text2music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <h1 align="center">🎵 YuE 2.0 (YuE2) Text-to-Music </h1>
<p align="center">
  <img src="https://img.shields.io/badge/Model-YuE2--3B--FP16-ff69b4?style=for-the-badge&logo=huggingface" />
  <img src="https://img.shields.io/badge/Backend-ComfyUI%20Native-blue?style=for-the-badge" />
  <img src="https://img.shields.io/badge/Hardware-T4%20%7C%20A100%20%7C%20L4%20GPU-green?style=for-the-badge&logo=nvidia" />
</p>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

###

---
### ⚡ Quickstart Guide
1. **Runtime Verification**: Ensure **Runtime** &rarr; **Change runtime type** &rarr; **T4 GPU** (or better) is active.
2. **Run Step 1 to 3**: Install the environment, download the checkpoint, and launch the backend daemon.
3. **Use the Interactive Studio (Step 4)**: Customize prompts, style tags, lyrics, durations, seeds, and samplers using dropdowns and sliders, then generate and stream audio directly in the cell.

In [ ]:
# @title 1. Hardware Verification & GPU Diagnostics
# @markdown Checks GPU memory and CUDA capabilities.
import torch
import sys
from IPython.display import HTML, display

if not torch.cuda.is_available():
    display(HTML("<div style='background-color:#ffebee;padding:12px;border-radius:8px;border-left:5px solid #f44336;'><h3 style='color:#c62828;margin:0;'>⚠️ GPU Not Detected!</h3><p style='margin:5px 0 0 0;'>Please navigate to <b>Runtime &rarr; Change runtime type</b> and select <b>T4 GPU</b> or higher.</p></div>"))
    raise SystemExit("GPU accelerator missing.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
cuda_ver = torch.version.cuda

display(HTML(f"""
<div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
  <h3 style='color:#2e7d32;margin:0 0 8px 0;'>✅ Hardware Accelerator Ready</h3>
  <table style='font-family:monospace;font-size:13px;'>
    <tr><td><b>GPU Device:</b></td><td>&nbsp;{gpu_name}</td></tr>
    <tr><td><b>Total VRAM:</b></td><td>&nbsp;{vram_gb} GB</td></tr>
    <tr><td><b>CUDA Version:</b></td><td>&nbsp;{cuda_ver}</td></tr>
    <tr><td><b>PyTorch:</b></td><td>&nbsp;{torch.__version__}</td></tr>
  </table>
</div>
"""))

In [ ]:
# @title 2. Install ComfyUI Core & Audio Dependencies
# @markdown Clones ComfyUI, installs audio libraries, and adds WebSocket client support for real-time telemetry.
import os
from IPython.display import HTML, display

print("🚀 Setting up ComfyUI audio environment...")

if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q soundfile torchaudio accelerate sentencepiece mutagen websocket-client safetensors
!apt-get -y install -qq aria2 ffmpeg > /dev/null 2>&1

display(HTML("<div style='background-color:#e3f2fd;padding:12px;border-radius:8px;border-left:5px solid #2196f3;'><b style='color:#1565c0;'>✅ Environment installed and ready!</b></div>"))

In [ ]:
# @title 3. Download FP16 Model & Launch Fast Daemon
# @markdown Downloads pre-converted `yue2_3b_fp16.safetensors` directly from Jokality/YuE2-fp16 and starts ComfyUI.
import os
import time
import subprocess
import urllib.request
from IPython.display import HTML, display

%cd /content/ComfyUI

model_dir = "/content/ComfyUI/models/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "yue2_3b_fp16.safetensors")

# Download pre-converted FP16 Checkpoint directly (no conversion needed)
if not os.path.exists(model_path) or os.path.getsize(model_path) < 6_000_000_000:
    print("📥 Downloading YuE2 FP16 checkpoint (~7.26 GB) from Jokality/YuE2-fp16... Please wait.")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
      "https://huggingface.co/Jokality/YuE2-fp16/resolve/main/checkpoints/yue2_3b_fp16.safetensors" \
      -d "{model_dir}" \
      -o "yue2_3b_fp16.safetensors"
else:
    print("⚡ Found existing FP16 checkpoint in cache.")

# Terminate previous instances to avoid port conflicts
!pkill -9 -f "main.py" || true
time.sleep(2)

print("🔄 Starting ComfyUI Turbo Daemon with native FP16 acceleration...")
server_log = open("/content/comfy_server.log", "w")

server_cmd = [
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--force-fp16",
    "--highvram",
    "--dont-upcast-attention"
]

proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=server_log, cwd="/content/ComfyUI")

server_alive = False
for i in range(40):
    if proc.poll() is not None:
        break
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        server_alive = True
        break
    except Exception:
        time.sleep(1.5)

if server_alive:
    display(HTML("""
    <div style='background-color:#e8f5e9;padding:15px;border-radius:8px;border-left:5px solid #4caf50;'>
      <b style='color:#2e7d32;'>🟢 YuE2 Native FP16 Engine Online!</b><br>
      <span style='color:#555;'>Model: <code>yue2_3b_fp16.safetensors</code> &bull; Hardware Tensor Cores Active. Run <b>Cell 4</b>.</span>
    </div>
    """))
else:
    print("\n❌ Startup failed. Log contents (/content/comfy_server.log):\n" + "-"*60)
    with open("/content/comfy_server.log", "r") as f:
        print(f.read())
    print("-" * 60)

In [ ]:
# @title 4. YuE 2.0 Studio Master (Interactive Music Generation)
# @markdown Configure style, lyrics, musical structure, and diffusion sampling parameters below.
import json
import time
import random
import glob
import os
import uuid
import subprocess
import websocket
import urllib.request
from IPython.display import HTML, display

# =========================================================================
# 🎚️ PARAMETER FORM CONTROLS
# =========================================================================

# @markdown ### 🎼 1. Style & Lyrics Prompt
genre_preset = "Indie Pop Anthem" #@param ["Custom", "Indie Pop Anthem", "Synthwave 80s Retro", "Lo-Fi Chill Hop", "Cinematic Epic Orchestral", "Modern Alt Rock"]
custom_style = "Upbeat indie pop with warm female vocals, bright electric guitars, punchy drums, melodic bass, and subtle synth layers. Catchy and energetic, with an uplifting summer atmosphere, a memorable chorus, and polished modern production." #@param {type:"string"}

lyrics = "[Verse]\nMorning light across the window\nCity waking down below\nI can hear the streets are calling\nFeels like somewhere we should go\n\n[Chorus]\nRun with me into the sunlight\nLeave the shadows far behind\nWe don't need to know tomorrow\nTonight the whole world feels alive\n\n[Verse]\nRadio playing through the open door\nLaughing like we did before\nEvery mile becomes a memory\nAnd I just want a little more\n\n[Chorus]\nRun with me into the sunlight\nLeave the shadows far behind\nWe don't need to know tomorrow\nTonight the whole world feels alive" #@param {type:"string"}

# @markdown ### ⚡ 2. Performance & ABC Planning
# @markdown *Disable `enable_abc_planning` to bypass the symbolic LLM score phase and synthesize direct audio (saves 2-3 minutes).*
# @markdown *Song length is controlled by your lyrics structure (no duration slider): ~1 min = Verse + Chorus + Outro, ~2 min = 2x Verse/Chorus + Outro, 3+ min = add a Bridge. End lyrics with `[Outro]` so the song closes naturally.*
enable_abc_planning = False #@param {type:"boolean"}
composition_mode = "full" #@param ["full", "melody"]
max_abc_tokens = 8192 #@param [1024, 2048, 4096, 8192] {type:"raw"}

# @markdown ### 🎲 3. Seed & Sampler
seed = 42 #@param {type:"integer"}
randomize_seed = True #@param {type:"boolean"}
sampler_name = "dpm_2" #@param ["dpm_2", "euler", "dpmpp_2m", "heun", "dpmpp_sde"]
scheduler = "sgm_uniform" #@param ["sgm_uniform", "normal", "karras", "simple"]
steps = 32 #@param {type:"slider", min:10, max:50, step:1}

# @markdown ### 💾 4. Audio Export
audio_format = "mp3" #@param ["mp3", "wav", "flac"]
audio_quality = "320k" #@param ["320k", "256k", "192k", "128k"]
filename_prefix = "audio/YuE2_Studio" #@param {type:"string"}

# =========================================================================
# 🔄 WORKFLOW LOGIC
# =========================================================================
preset_dict = {
    "Indie Pop Anthem": "Upbeat indie pop with warm female vocals, bright electric guitars, punchy drums, melodic bass, and subtle synth layers. Catchy and energetic, with an uplifting summer atmosphere, a memorable chorus, and polished modern production.",
    "Synthwave 80s Retro": "1980s retro synthwave, analog vintage synthesizers, gated snare reverb, driving arpeggiated bassline, male nostalgic vocals, neon aesthetic, cinematic nocturnal drive.",
    "Lo-Fi Chill Hop": "Warm lofi hip hop, vinyl crackle, mellow electric piano chords, smooth relaxed boom bap drums, deep warm sub bass, peaceful afternoon coffee shop atmosphere.",
    "Cinematic Epic Orchestral": "Epic cinematic trailer music, soaring brass ensemble, thundering Taiko drums, expressive emotional strings, dramatic crescendos, heroic orchestral climax.",
    "Modern Alt Rock": "Modern alternative rock, overdriven crunchy guitars, aggressive live acoustic drum kit, energetic lead vocals, heavy bassline, anthemic chorus."
}

# Ceiling only (matches the official ComfyUI template). YuE2 ends the song on its own when the lyrics
# are finished, so this is NOT a target length and should stay high to avoid cutting songs mid-section.
MAX_DURATION_SECONDS = 360

active_style = custom_style if genre_preset == "Custom" else preset_dict.get(genre_preset, custom_style)
active_seed = random.randint(1, 2147483647) if randomize_seed else seed
ckpt_name = "yue2_3b_fp16.safetensors"

workflow_api = {
  "10": {
    "inputs": {
      "filename_prefix": filename_prefix,
      "format": audio_format,
      "format.quality": audio_quality,
      "audio": ["33:9", 0]
    },
    "class_type": "SaveAudioAdvanced"
  },
  "33:15": {
    "inputs": {"ckpt_name": ckpt_name},
    "class_type": "CheckpointLoaderSimple"
  },
  "33:24": {
    "inputs": {
      "style": ["33:36", 0],
      "lyrics": ["33:37", 0],
      "seed": ["33:34", 0],
      "mode": composition_mode,
      "max_abc_tokens": max_abc_tokens,
      "temperature": 0.7,
      "top_p": 0.9,
      "top_k": 30,
      "repetition_penalty": 1.005,
      "penalty_window": 100,
      "clip": ["33:15", 1]
    },
    "class_type": "YuE2GenerateABC"
  },
  "33:14": {
    "inputs": {"source": ["33:30", 0]},
    "class_type": "PreviewAny"
  },
  "33:30": {
    "inputs": {
      "switch": ["33:31", 0],
      "on_false": ["33:32", 0],
      "on_true": ["33:24", 0]
    },
    "class_type": "ComfySwitchNode"
  },
  "33:32": {
    "inputs": {"value": ""},
    "class_type": "PrimitiveString"
  },
  "33:25": {
    "inputs": {
      "style": ["33:36", 0],
      "lyrics": ["33:37", 0],
      "abc": ["33:14", 0],
      "seed": ["33:34", 0],
      "mode": composition_mode,
      "max_duration": MAX_DURATION_SECONDS,
      "temperature": 1.0,
      "top_p": 0.95,
      "top_k": 100,
      "repetition_penalty": 1.2,
      "clip": ["33:15", 1]
    },
    "class_type": "YuE2GenerateMusic"
  },
  "33:31": {
    "inputs": {"value": enable_abc_planning},
    "class_type": "PrimitiveBoolean"
  },
  "33:18": {
    "inputs": {"conditioning": ["33:25", 0]},
    "class_type": "ConditioningZeroOut"
  },
  "33:23": {
    "inputs": {"source": ["33:25", 1]},
    "class_type": "PreviewAny"
  },
  "33:8": {
    "inputs": {
      "seed": ["33:34", 0],
      "steps": steps,
      "cfg": 1.0,
      "sampler_name": sampler_name,
      "scheduler": scheduler,
      "denoise": 1.0,
      "model": ["33:15", 0],
      "positive": ["33:25", 0],
      "negative": ["33:18", 0],
      "latent_image": ["33:5", 0]
    },
    "class_type": "KSampler"
  },
  "33:5": {
    "inputs": {
      "seconds": ["33:25", 1],
      "batch_size": 1
    },
    "class_type": "EmptyYuE2LatentAudio"
  },
  "33:9": {
    "inputs": {
      "samples": ["33:8", 0],
      "vae": ["33:15", 2]
    },
    "class_type": "VAEDecodeAudio"
  },
  "33:34": {
    "inputs": {"seed": active_seed},
    "class_type": "SeedNode"
  },
  "33:36": {
    "inputs": {"value": active_style},
    "class_type": "PrimitiveStringMultiline"
  },
  "33:37": {
    "inputs": {"value": lyrics},
    "class_type": "PrimitiveStringMultiline"
  }
}

node_names = {
    "33:15": f"CheckpointLoaderSimple ({ckpt_name})",
    "33:24": "YuE2GenerateABC (Symbolic Score Planner)",
    "33:30": "ComfySwitchNode (ABC Bypass Route)",
    "33:25": "YuE2GenerateMusic (Acoustic Audio Synthesis)",
    "33:8": f"KSampler (Diffusion Decoding - {steps} steps)",
    "33:9": "VAEDecodeAudio (Audio Decompression)",
    "10": "SaveAudioAdvanced (MP3 Export)"
}

# Display job card
display(HTML(f"""
<div style='background-color:#1e1e2f;color:#fff;padding:16px;border-radius:10px;margin-bottom:15px;'>
  <h3 style='margin:0 0 10px 0;color:#ff79c6;'>🚀 YuE2 Job Dispatched</h3>
  <div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:10px;font-size:13px;'>
    <div><b>Model:</b> <span style='color:#50fa7b;'>{ckpt_name}</span></div>
    <div><b>ABC Planning:</b> <span style='color:#f1fa8c;'>{'Enabled' if enable_abc_planning else 'Bypassed (Direct Audio)'}</span></div>
    <div><b>Length:</b> <span style='color:#8be9fd;'>Auto (follows lyrics, max {MAX_DURATION_SECONDS}s)</span></div>
    <div><b>Sampler:</b> <span style='color:#bd93f9;'>{sampler_name} ({steps} steps)</span></div>
  </div>
</div>
"""))

client_id = str(uuid.uuid4())
ws = websocket.WebSocket()
ws.connect(f"ws://127.0.0.1:8188/ws?clientId={client_id}")

req_payload = json.dumps({"prompt": workflow_api, "client_id": client_id}).encode('utf-8')
req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=req_payload, headers={'Content-Type': 'application/json'})
resp = urllib.request.urlopen(req)
prompt_id = json.loads(resp.read())['prompt_id']

print(f"📡 [Prompt ID: {prompt_id}] Active Seed: {active_seed}")
print("=" * 65)

start_time = time.time()
job_failed = False

while True:
    msg = ws.recv()
    if isinstance(msg, str):
        event = json.loads(msg)
        event_type = event.get('type')
        event_data = event.get('data', {})

        # Capture errors in the pipeline
        if event_type == 'execution_error':
            print(f"\n❌ Pipeline Error in Node {event_data.get('node_id')}: {event_data.get('exception_message')}")
            job_failed = True
            break

        if event_type == 'executing':
            node_id = event_data.get('node')
            if node_id is None:
                if event_data.get('prompt_id') == prompt_id:
                    print("\n\n🎉 Pipeline Completed!")
                    break
            else:
                stage_label = node_names.get(str(node_id), f"Node {node_id}")
                print(f"\n▶️ [Stage: {stage_label}]")

        elif event_type == 'progress':
            curr_val = event_data.get('value', 0)
            max_val = event_data.get('max', steps)
            pct = int((curr_val / max_val) * 100)
            bar_len = 25
            filled = int(bar_len * curr_val // max_val)
            bar = '█' * filled + '░' * (bar_len - filled)
            elapsed_s = int(time.time() - start_time)
            print(f"\r   Sampling: [{bar}] {curr_val}/{max_val} ({pct}%) | {elapsed_s}s elapsed", end="", flush=True)

ws.close()
total_seconds = round(time.time() - start_time, 1)

# Display player card
if not job_failed:
    candidate_files = glob.glob(f"/content/ComfyUI/output/{filename_prefix}*.{audio_format}")
    if not candidate_files:
        candidate_files = glob.glob(f"/content/ComfyUI/output/audio/*.{audio_format}")

    if candidate_files:
        latest_file = max(candidate_files, key=os.path.getctime)
        file_size_mb = round(os.path.getsize(latest_file) / (1024 * 1024), 2)
        filename_base = os.path.basename(latest_file)

        try:
            _d = float(subprocess.check_output(
                ["ffprobe","-v","error","-show_entries","format=duration","-of","default=nw=1:nk=1", latest_file]).strip())
            actual_duration_txt = f"{int(_d//60)}:{int(_d%60):02d}"
        except Exception:
            actual_duration_txt = "n/a"

        import base64
        with open(latest_file, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode("utf-8")

        mime_type = "audio/mpeg" if audio_format == "mp3" else f"audio/{audio_format}"

        display(HTML(f"""
        <div style='background:#121214;border:1px solid #27272a;border-radius:14px;padding:22px;max-width:680px;font-family:sans-serif;color:#f4f4f5;box-shadow:0 10px 25px rgba(0,0,0,0.5);margin-top:15px;'>
          <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:14px;'>
            <div style='display:flex;align-items:center;gap:10px;'>
              <div style='background:linear-gradient(135deg,#ec4899,#8b5cf6);width:42px;height:42px;border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:22px;'>🎵</div>
              <div>
                <h3 style='margin:0;font-size:16px;color:#fff;'>{filename_base}</h3>
                <span style='color:#a1a1aa;font-size:12px;'>Rendered in <b>{total_seconds}s</b> &bull; {file_size_mb} MB &bull; {actual_duration_txt} Audio</span>
              </div>
            </div>
            <span style='background:#22c55e20;color:#4ade80;padding:4px 10px;border-radius:20px;font-size:11px;font-weight:bold;'>READY</span>
          </div>

          <audio controls autoplay style='width:100%;margin:12px 0;border-radius:8px;'>
            <source src='data:{mime_type};base64,{audio_b64}' type='{mime_type}'>
          </audio>

          <div style='background:#18181b;padding:12px;border-radius:8px;font-size:12px;margin-top:10px;border:1px solid #27272a;'>
            <div><b>Style:</b> <span style='color:#cbd5e1;'>{active_style[:140]}...</span></div>
            <div style='display:flex;justify-content:space-between;margin-top:6px;'>
              <span><b>Seed:</b> <code style='color:#38bdf8;'>{active_seed}</code></span>
              <span><b>Sampler:</b> {sampler_name} / {scheduler}</span>
              <span><b>Bitrate:</b> {audio_quality}</span>
            </div>
          </div>
        </div>
        """))
    else:
        print("⚠️ No audio output found in /content/ComfyUI/output/")

In [ ]:
# @title 5. File Manager: Download All Generated Tracks
# @markdown Compresses all audio files in the output directory into a single `.zip` and prompts a browser download.
import glob
import os
import shutil
from google.colab import files

output_dir = "/content/ComfyUI/output/audio"
all_audio = glob.glob(f"{output_dir}/*.*")

if not all_audio:
    print("No audio tracks generated yet.")
else:
    print(f"Found {len(all_audio)} track(s):")
    for track in all_audio:
        size_mb = round(os.path.getsize(track) / (1024*1024), 2)
        print(f" - {os.path.basename(track)} ({size_mb} MB)")

    zip_name = "/content/YuE2_Generated_Tracks"
    shutil.make_archive(zip_name, 'zip', output_dir)
    print("\n📦 Downloading tracks zip to your local computer...")
    files.download(f"{zip_name}.zip")